# YLM v1: run training packs on Colab

1. Runtime -> Change runtime type -> **GPU** (T4 is fine).
2. Put `ylm_data.zip` (made by `colab/make_bundle.sh`; raw data files are git-ignored) into **MyDrive/ylm/** on Google Drive.
3. Run the cells in order. The code is cloned from GitHub (`main` by default). Pick one `PACK`; each line of a pack file is one training.
   Results are synced to **MyDrive/ylm/runs/** every 5 minutes.

Packs: `v1_0 ... v1_3` = main interface table (seeds 0-2: bc, optA, prog, rule, oracle), `v1_4 ... v1_7` = ablations,
`pol` = polarizability (3 seeds), `quad_s0..2` = l=2 quadrupole.

In [ ]:
import subprocess
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, subprocess, sys, zipfile
BRANCH = os.environ.get('YLM_BRANCH', 'main')
REPO_URL = os.environ.get('YLM_REPO_URL', 'https://github.com/yaniguan/YLM.git')
DATA = '/content/drive/MyDrive/ylm/ylm_data.zip'
assert os.path.exists(DATA), 'upload ylm_data.zip to MyDrive/ylm/ first'
if not os.path.exists('/content/YLM'):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, '/content/YLM'], check=True)
os.chdir('/content/YLM')
print(subprocess.check_output(['git', 'log', '--oneline', '-1'], text=True))
os.makedirs('data', exist_ok=True)
with zipfile.ZipFile(DATA) as z:
    z.extractall('.')
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'e3nn==0.6.0', 'scipy'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_v1_ported_models.py'], check=True)


In [ ]:
PACK = 'jobs/packs/v1_0.txt'
N_PARALLEL = 3
EXTRA = ''
print(open(PACK).read())


In [ ]:
import os, subprocess, sys, threading, time
os.makedirs('runs', exist_ok=True)
DRIVE_RUNS = '/content/drive/MyDrive/ylm/runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
stop = False
def sync():
    while not stop:
        subprocess.run(['rsync', '-a', 'runs/', DRIVE_RUNS + '/'])
        time.sleep(300)
threading.Thread(target=sync, daemon=True).start()
cmd = [sys.executable, 'jobs/run_pack.py', '--pack', PACK, '--parallel', str(N_PARALLEL)]
if EXTRA:
    cmd += ['--extra', EXTRA]
t0 = time.time()
subprocess.run(cmd, check=True)
stop = True
subprocess.run(['rsync', '-a', 'runs/', DRIVE_RUNS + '/'])
print('all done in %.1f h' % ((time.time() - t0) / 3600))


In [ ]:
import glob, os
for f in sorted(glob.glob('runs/*.stdout')):
    lines = [l for l in open(f) if l.startswith('epoch') or 'Error' in l]
    print(os.path.basename(f), '|', (lines[-1].strip()[:150] if lines else 'no epoch yet'))
